# Нырок в Си

В этой заметке пробегусь по ["основам языка \[Cython\]"](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html) с большим погружением в Си. Оставляю статейку по [основам языка Си](https://thecode.media/osnovy-sintaksisa-yazyka-c/), которых может хватить.

In [1]:
%load_ext cython

### Байт с характером

Если задать вопрос "Что такое байт", скорее всего ответят ["это 8 бит"](https://en.wikipedia.org/wiki/Byte). Это верно для большинства систем, но в целом неверно, потому что, простыми словами, "[байт](https://stackoverflow.com/questions/8296426/what-is-a-byte-in-c-c) - это наименьшая (минимальная) адресуемая ячейка памяти, в которой гарантированно можно хранить один символ из стандартного набора символов (например, ASCII или его расширений) для данной платформы". Когда Си появился, в его системе типов не определили тип `byte`, но из-за возможности представлять символ определили тип `char` ("character" - символ), поэтому можно встретить название "символьный тип". Важно понимать, что:

- символ - это целочисленный код, а значит char хранит целые (integer) числа;
- размер char - это байт, а вот сколько бит он представляет можно посмотреть в макросе **CHAR_BIT**;
- char может представляться как `signed char` либо `unsigned char` - зависит от платформы.

In [2]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `#include <limits.h>`
cimport libc.limits as limits

print(f"CHAR_BIT = {limits.CHAR_BIT}")

cdef signed char c = 'A'
cdef unsigned char uc = c  # safe implicit conversion
print(f"c = {c}; uc = {uc}")
print(f"c + 1 = {chr(c + 1)}; uc - 1 = {chr(uc - 1)}")

c = limits.CHAR_MAX
uc = limits.UCHAR_MAX
print(f"CHAR_MAX = {c}; UCHAR_MAX = {uc}")

# overflow(s)
c += 1   # UB = undefined behaviour
uc += 1  # DB = defined behaviour
print(f"Overflow(s): c={c} and uc={uc}")

CHAR_BIT = 8
c = 65; uc = 65
c + 1 = B; uc - 1 = @
CHAR_MAX = 127; UCHAR_MAX = 255
Overflow(s): c=-128 and uc=0


В примере выше пользуются [внешние](https://cython.readthedocs.io/en/latest/src/tutorial/external.html) сишные макросы благодаря сайтоновскому `cimport`. Сайтон распознал, что пайтоновская строка (ведь строки задаются и одиночными кавычками) с одним символом впишется в [тип char](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#types). Помним, что символы - это коды, поэтому символ 'A' представляется целочисленным кодом 65 (подробнее в [man ascii](https://man7.org/linux/man-pages/man7/ascii.7.html)).

И куда без переполнения и продвижения типов (type promotion) - то, о чём в Пайтоне не просто не принятно, а нет нужды помнить.

In [3]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `#include <limits.h>`
cimport libc.limits as limits

cdef signed char c = limits.CHAR_MAX
cdef unsigned char uc = limits.UCHAR_MAX
print(f"CHAR_MAX = {c}; UCHAR_MAX = {uc}")

# type promotion
print(f"c + 1 = {c + 1} -> promoted to int ({sizeof(c + 1)} bytes)")
print(f"uc + 1 = {uc + 1} -> promoted to int ({sizeof(uc + 1)} bytes)")

# overflow(s)
c += 1   # UB = undefined behaviour
uc += 1  # DB = defined behaviour
print(f"Overflow(s): [c += 1 -> {c}] and [uc += 1 -> {uc}]")

CHAR_MAX = 127; UCHAR_MAX = 255
c + 1 = 128 -> promoted to int (4 bytes)
uc + 1 = 256 -> promoted to int (4 bytes)
Overflow(s): [c += 1 -> -128] and [uc += 1 -> 0]


Пояснения:

- Для выражений `c + 1` и `uc + 1` происходит продвижение типов:
  - литерал (буквальное/literal значение) 1 имеет тип `int` (проверьте);
  - младшие символьные типы "подтягиваются" до старшего целого (int) и получается, что в обоих случаях имеет место `int + int`;
  - это нужно, чтобы приравненно (адекватно) представить выдаток (рузультат) сложения чисел;
  - если я правильно понял, то продвижение к `short` не происходит, потому что ранее компиляторы и так приводили к `int` как к "естественному" целевому целочисленному типу, да и нет ручательства, что `short` меньше `int` - он точно не больше, так что может быть и равен.

- Переполнение типов воспроизведено через присваивания:
  - `x += 1` - это `x = x + 1` - не в точности так (см. на уровне инструкицй), но в основном можно считать так;
  - сначала продвижение типов, затем усечение от большего типа к меньшему и это не безопасно, но проканало:
    - переполнение знакового типа есть неопределённое поведение (undefined behaviour);
    - для знакового определён круговой (циклический) сдвиг, или же "провёртка" (так я решил передать "wrap-around") - это уже определённое/ожидаемое поведение.

Предполагаю, что читатель(ница) знает за целые и "плавающие" (float) числа (и даже читал(а) стандарт [IEEE 754](https://ru.wikipedia.org/wiki/IEEE_754)) ибо это не курс по языку Си и устройству вычислительных (компьютерных) систем (мне просто лень оттаптываться), поэтому сразу направимся к хорошим собраниям.

### Массивы

Массив - собрание (коллекция) для хранения однородных (гомогенных) элементов. Сайтон поддерживает два стиля объявления [массивов](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#c-arrays): Java и С. Легче и проще пользоваться "явовским" стилем, потому что он поддерживает объявление и "обначаливание" (инициализацию) в одну строку, тогда как для Си-стиля такое не прокатывает, считается непоощряемым и приводит к ошибке (на моей машине точно).

In [4]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `from libc.math cimport sin`
cdef extern from "math.h":
    double sin(double x)


# C-style -> declare and initialise in one line
# cdef float point[3] = [0.1, -0.2, 0.3]  # Syntax error in C variable declaration

# C-style -> declare and then initialise separately -> possible yet discouraged
# cdef float point[3]
# point = [0.1, -0.2, 0.3]

# Java-style
cdef float[3] point = [0.1, -0.2, 0.3]
print(f"3D point coords -> {point}")

sines = [sin(point[i]) for i in range(len(point))]  # Python list
print(f"sines of the point -> {sines}")

3D point coords -> [0.10000000149011612, -0.20000000298023224, 0.30000001192092896]
sines of the point -> [0.0998334181294999, -0.19866933371588721, 0.29552021804983797]


Вместо сайтоновского импорта `from libc.math cimport sin` я решил попробовать "включить" (include) прототип функции `sin` из заголовочника (заголовочного файла - header file) [math.h](https://man7.org/linux/man-pages/man0/math.h.0p.html). Этот заголовочник стандартный, а значит он и его исходник (исходный файл - source file) известны системе и нет нужды "наставлять" компилятор по каким путям искать математические функции.

Массив занимает некоторую область памяти, которая имеет начало и конец. Эта область памяти - ряд ячеек памяти (байтов), но в вышнем примере массив хранит "плавающие" (float) числа, а такое число может занимать, например, 4 байта. Получается, что массив содержит три "ячейки" (пока назову их так), и каждая ячейка массива состоит из 4х ячеек памяти (байтов), поэтому для массива из трёх "плавающих" чисел отводится 12 байт, т.е. "3 (число элементов) * 4 (размер типа данных)". Да что толковать, можно посмотреть:

In [5]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from cython cimport sizeof

cdef float[3] point = [0.1, -0.2, 0.3]

print(f"sizeof(float) = {sizeof(float)}")
print(f"sizeof(point) = {sizeof(point)}")
print(f"the length of point = {sizeof(point) // sizeof(float)}")

sizeof(float) = 4
sizeof(point) = 12
the length of point = 3


Видно, что в общем случае **"ячейка массива" не всегда то же, что и "ячейка памяти"**, хотя равенство возможно если это массив символов/байтов. Поэтому размер массива в памяти вычисляется по формуле `array_size * sizeof(item_type)`.

Я не зря веду речь за память и её области (regions) уже сейчас, потому что в Си работать с памятью приходится довольно часто, отсюда важно понимание её устройства и как с ней следует управляться. Именно поэтому "с сейчас" я начинаю ранний подступ к одному из мощнейших средств низкоуровневого программирования - указатели.

### Указатели

Указатели (pointers) - та тема, на которой Си указал мне на дверь, когда я пытался его учить самостоятельно. Когда я проходил бассейн в [Школе 21](https://21-school.ru/) (тогда ещё франшиза [École 42](https://42.fr/en/homepage/)), мне пришлось их освоить чтобы выжить и проплыть до победного завершения. В Пайтоне нет нужды в плотном соприкасании с памятью, а значит и указатели как средство языка убраны от греха подальше (а нагрешить легко). При этом мериловый (эталонный) Python написан как раз на языке Си (отсюда название его реализации [CPython](https://github.com/python/cpython)), а значит можно ожидать, что нутро CPython очень даже кишит указателями как [Фрай червями](https://ru.wikipedia.org/wiki/Parasites_Lost).

Простой (ну правда же) пример.

In [6]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdint cimport uintptr_t

cdef unsigned char uc = 'W'
print(f"uc = {uc} ('{chr(uc)}')")

# ptr is a variable of type `pointer (*) to an unsigned char` 
cdef unsigned char * ptr
# now ptr stores the address (&) of the variable `uc`
ptr = &uc
print(f"the address of uc (`&uc`) is {(<uintptr_t>ptr):#x}")

# using `*ptr` leads to error, `ptr[0]` saves the cell
print(f"the value of `*ptr` is {ptr[0]}")

uc = 87 ('W')
the address of uc (`&uc`) is 0x7f0dc8531390
the value of `*ptr` is 87


Вот как можно этот пример написать на Си:

```c
#include <stdio.h>

int main() {
	unsigned char uc = (unsigned char) 'W';  // type casting from `int`
	printf("uc = %hhu (%c)\n", uc, uc);

	unsigned char *ptr;
    ptr = &uc;

	printf("the address of uc (`&uc`) is %p\n", ptr);
	printf("the value of `*ptr` is %c\n", *ptr);

	return 0;
}
```

Разбор примера (для тех, кто пока не очень с указателями + повторение - мать учения):

- объявлена переменная `unsigned char uc`, которой присвоен символ 'W' (код 87);
- эта переменная именует объект памяти - байт со значением 87 (ему хватит ячейки ибо байт);
- объявляю переменную `ptr`, тип которой - указатель/pointer (помечается `*`) на `unsigned char`, которой я присваиваю адрес (местоположение), переменной `uc` с помощью `&uc`
  -  `&uc` - это взятие адреса переменной `uc`, а не значения по ней
  -  **переменная-указатель хранит некий адрес** означает, что **эта переменная указывает на данные по этому адресу**
  -  с помощью оператора разыменования (dereferencing) можно получить доступ к данным по адресу, т.е. `*ptr` (здесь это также достижимо через `*(&uc)` - здесь звёздочка именно про доступ данных, а не про объявление указателя как это было раньше.
  - Сайтон оказался не без приколов: вместо сишного `*ptr` пришлось писать `ptr[0]`.

Но почему пришлось писать `ptr[0]` вместо сишечного `*ptr`?! Откуда индексация если ptr не массив?! Потому что массивы и указатели связаны теснее и под покровами синтаксиса скрывается адресная арифметика, благодаря которой `ptr[0] = *(ptr + 0) = *ptr`. Поясню на примере массива, пусть с именем `arr`:

- `arr[0]` обращается к элементу с нулевым смещением от начала массива, т.е. первый элемент, потому что `arr` "указывает" на границу начала области памяти, т.е. содержит её адрес;
- адрес - это число, а значит `arr = arr + 0`, поэтому нет разницы как перейти по этому адресу, ведь `*(arr + 0) == *arr`;
- но это и значит, что `arr[0] == *(arr + 0) == *arr`, а теперь подставь вместо `arr` имя `ptr` и узри "однотожесть"!

In [7]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdint cimport uintptr_t

ctypedef int * intptr_t

cdef int[4] nums = [1, -2, 3, -4]

# same as `cdef int * ptr = nums`
cdef intptr_t ptr = nums

print(f"the address of nums (ptr) is {(<uintptr_t>nums):#x}\n")

for i in range(len(nums)):
    print(f"nums[{i}] = {nums[i]} <-> ptr[{i}] == {ptr[i]}")

the address of nums (ptr) is 0x7f0dc852a3e0

nums[0] = 1 <-> ptr[0] == 1
nums[1] = -2 <-> ptr[1] == -2
nums[2] = 3 <-> ptr[2] == 3
nums[3] = -4 <-> ptr[3] == -4


Хорошо, что удалось показать использование `ctypedef` (посмотреть применение можно [здесь](https://cython.readthedocs.io/en/latest/src/tutorial/strings.html)) для создания псевдонимов типов: так, вместо типа `int *` можно писать `intptr_t`. Да, получилось длиннее и да, можно было использовать `intptr_t` как раз из библиотеки `libc.stdint`, но для образовательного показа вполне сойдёт.

Как массивы связаны с указателями, а указатели с памятью, самое время проложить путь к [памятным местам](https://ru.wikipedia.org/wiki/%D0%9C%D0%B5%D1%82%D0%BE%D0%B4_%D0%BB%D0%BE%D0%BA%D1%83%D1%81%D0%BE%D0%B2).

### Игры с памятью

Если процессу (грубо, исполняемой программе) потребуется дополнительная память, нужно попросить ОС выделить ценный ресурс и возможно он будет предоставлен (ручательств/гарантий нет). Это называется [динамическим распределением памяти](https://ru.wikipedia.org/wiki/%D0%94%D0%B8%D0%BD%D0%B0%D0%BC%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D1%80%D0%B0%D1%81%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BF%D0%B0%D0%BC%D1%8F%D1%82%D0%B8), потому что память выделяется процессу во время его исполнения, в его динамике (не статике).

Здесь поиграюсь с двумя полезными функциями:

- [malloc](https://man7.org/linux/man-pages/man3/malloc.3.html) - выделить (allocate) память процессу;
- [free](https://man7.org/linux/man-pages/man3/free.3p.html) - освободить ранее выделенную память.

Да, затребованную память нужно освобождать, чтобы избежать [утечек памяти](https://coddy.tech/docs/ru/c/memory-leaks) - коварного и опасного способа привести процесс и систему у отказу из-за исчерпания общей (!) памяти, которой пользуются и другие процессы.

In [8]:
%%cython

#-c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport malloc, free

cdef unsigned char arr_len = 4
print(f"array length = {arr_len}")

# sizeof(char) is better than hardcoding it as 1
cdef unsigned char item_size = sizeof(char)
print(f"sizeof(char) = {item_size} (byte[s])")

# Explicit type conversion is vital here -> Cannot assign type 'void *' to 'char *'
cdef char *s = <char *>malloc(arr_len * item_size)

# malloc may fail -> the best practice is to always check its return value
if s is NULL:
    raise MemoryError("malloc failed") from None

cdef size_t idx = 0
cdef char ch
# try to comment cdef for idx and ch and see the failure
for idx, ch in enumerate(b"abc"):
    s[idx] = ch
    idx += 1

# slices are doable!
print(f"array = {s[:idx]}")

# allocated memory must be free'd (returned to the system)
free(s)

array length = 4
sizeof(char) = 1 (byte[s])
array = b'abc'


Пример диалектический: он прост и непрост:

- `char` может быть знаковым или беззнаковым (зависит от архитектуры), а в примере был использован `sizeof(char)` - это видимая небрежность, ведь знаковость не влияет на размер самого типа (дополнительно на [StackOverflow](https://stackoverflow.com/questions/69810915/why-is-sizeof-a-char-literal-not-the-same-as-sizeofchar));
- в Си явное (explicit) приведение типа (type casting) возврата из malloc к типу `<char *>` избыточно, ведь переменная `s` уже имеет нужный тип и будет сделано неявное (implicit) приведение, но мы используем жёсткие флаги компиляции, из-за которых и требуется явное приведение, иначе: ___"Cannot assign type 'void *' to 'char *'"___, потому что malloc возвращает указатель на любой тип данных (`void *`), а это иной тип данных;
- возвраток от malloc нужно **обязательно проверять на NULL**, ведь ОС может не справиться и просто не выделить память, а разыменовывать NULL-указатель опасно поломками;;
- чтобы безопасно пропечатать массив, пришлось использовать срезы, а не просто подать объект (об этом чуть далее);
- по завершению, **обязательно зачищать/освобождать выделенную память** когда та становится невостребованной.

Казалось бы, пример простой: скопировать строку по новому месту в памяти и отдать на печать. Но если не использовать срезы, произойдёт выход за границы массива и выполнение ячейки и всей тетрадки может поломаться (а может и по системе шарахнет, зависит). Но откуда выход за границы? А снова особенности низкоуровневщины:

- массив в Си - это про область памяти, а не про структуру данных (хотя такую можно определить), а значит размер этой области нужно помнить/вычислять, чтобы точно знать где ходить можно, а где уже выход за границы
- в Си нет строк как типа данных, под строками в Си понимаются нуль-терминированные массивы символов (о боги!).

Если всё ясно и с глубин подсознания нахлынывают воспоминания о "чём-то подобном", это славно, а если неть, то и вовсе прекрасно, ведь разбору и поиску ответа посвящается следующая глава. 

### Строки

В Пайтоне (я не рассматриваю версии меньше Python3) есть тип строка [str](https://docs.python.org/3/builtins/stdtypes.html#text-sequence-type-str) - мощные, [юникодные](https://home.unicode.org/) и сверхполноценные. Также есть тип [bytes](https://docs.python.org/3/builtins/stdtypes.html#bytes) - байтовые последовательности, или байтовые строки. В языке Си **строк как отдельного типа данных нет**, а строкой считается *массив символов с завершающим (терминирующим) символом '\0' (NUL)*. Символ '\0' помечает конец "строки", поэтому он называется нуль-терминатором (NUL-terminator, т.е. нуль-завершитель), который [не то же самое, что NULL](https://stackoverflow.com/questions/1296843/what-is-the-difference-between-null-0-and-0).

Пример.

In [9]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport malloc, free
from libc.string cimport strcpy, strcat, strlen

# byte strings are closer to ASCII C strings
cdef const char* s1 = b"Hell\0o"  # not "Hello", but "Hell" because of '\0'
cdef const char* s2 = b"World"   # '\0' is not required by default

cdef size_t n = strlen(s1) + strlen(s2) + 1  # +1 byte to store '\0'
cdef char* s3 = <char*>malloc(n)

# malloc can fail, therefore it is vital to check the result
if s3 == NULL:
    raise MemoryError("malloc failed") from None

try:
    strcpy(s3, s1)  # writes s1 in the empty buffer s3
    strcat(s3, s2)  # (con)catenates s2 to the end of s1
    print(s3)
finally:
    free(s3)

b'HellWorld'


Использование байтовых строк необходимо для явного обеспечения [ASCII](https://ru.wikipedia.org/wiki/ASCII) режима во избежания международного скандала с участием Юникода. Строки **s1** и **s2** особенны тем, что их типы помечены [квалификатором](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#type-qualifiers) const, чтобы их нельзя было изменить наместно (in-place). А далее простой алгоритм:

- выделение памяти под новую строку (совокупная длина двух строк) с учётом '\0' знака/символа, отсюда и +1 байт;
- копирование строки s1 в буфер s3, т.е. область памяти, на которую ссылается переменная-указатель s3 - [strcpy](https://man7.org/linux/man-pages/man3/strcpy.3.html) добавил завершающий нуль-символ;
- сцепление (конкатенация) строки s2 с уже строкой s3 - уже [strcat](https://man7.org/linux/man-pages/man7/strcat.7.html) добавил нуль=завершитель от себя.

В этом примере кроется вообще-то неприличный недостаток: нет проверки на переполнение типов. Строки могут быть столь длинными, что их сложение или добавка всего одного байта может не влезть в size_t, а значит на самом деле выделенный объём памяти будет неприравненным (неадэкватным) и это уже ошибка. По-хорошему, такую проверку нужно сделать, но сие избыточно для этого примера и его поучительная часть такая: **не нужно усложнять там где это не нужно**. Ну видно же, что длины строк достаточно уместительные, можно и попроще. Также я не занулил (не присвоил NULL) переменной s3 - по тем же причинам, что это не сделает погоды в данном примере.

Разобранный пример можно доразобрать и сделать сайтонические подобия (!) функций `strcpy` и `strcat`.

In [10]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport realloc, free


cdef size_t ft_strlen(const char *s):
    cdef const char * start = s
    while s[0]:
        s += 1
    return s - start


cdef char * ft_strcpy(char * dest, const char * src):
    if not dest:
        return NULL
    if not src:
        return dest
    cdef char * start = dest
    while ch := src[0]:
        dest[0] = ch
        dest += 1
        src += 1
    dest[0] = b'\0'
    return start


cdef char * ft_strcat(char * dest, const char * src):
    if not dest:
        return NULL
    if not src:
        return dest
    cdef char * start = dest
    dest += ft_strlen(dest)
    while ch := src[0]:
        dest[0] = ch
        dest += 1
        src += 1
    dest[0] = b'\0'
    return start


# Here we go...


cdef const char* s1 = b"Hell\0o"
cdef const char* s2 = b"World"

cdef char* s3 = <char *>realloc(NULL, ft_strlen(s1) + ft_strlen(s2) + 1)
if s3 == NULL:
    raise MemoryError("realloc failed") from None

try:
    ft_strcpy(s3, s1)
    ft_strcat(s3, s2)
    print(s3)
finally:
    free(s3)
    s3 = NULL  # redundant here, OK in general

b'HellWorld'


Здесь решил ввести функцию [realloc](https://man7.org/linux/man-pages/man3/realloc.3p.html), которой передан `NULL` в качестве указателя на область памяти на перевыделение (reallocation) - в таком случае realloc ведёт себя как malloc. И важные замечания о free (а подробнее [здесь](https://habr.com/ru/companies/pvs-studio/articles/791798/)):

- free нужно передавать указатель на ранее выделенную область памяти из кучи, а не произвольный адрес;
- free нужно использовать один раз для некоторого адреса, повторное использование - неопределённое поведение;
- free не зануляет указатель, потому что принимает адрес по значению (copy-by-value), а не по ссылке (copy-by-reference)
- free может принимать NULL (возрадуемся).

Но можно написать и свою безопасную обёртку.

In [11]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdlib cimport realloc, free

cdef void free_safe(void **ptr):
    if target := ptr[0]:
        free(target)
        ptr[0] = NULL
        # target = NULL -> will update the local variable target!


cdef size_t arrsiz = 3
cdef int * coords = <int *>realloc(NULL, arrsiz * sizeof(int))
if not coords:
    raise MemoryError("realloc failed") from None

cdef int idx, num
for idx, num in enumerate([1, -2, 3]):
    coords[idx] = num

print(f"coords = {[coords[i] for i in range(arrsiz)]}")

free_safe(<void **>&coords)

if coords is NULL:
    print(f"coords are freed")

coords = [1, -2, 3]
coords are freed


Вполне неплохо получилось, можно заканчивать, и так покрыто достаточно.